<a href="https://colab.research.google.com/github/sukumar571/E-news/blob/main/DeepLure_Color_Invariant_Saree_Design_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepLure AI Engineer Assessment
## Color-Invariant Saree Design Recognition

**Goal:** Build an image-classification system that recognizes saree designs while reducing dependence on saree color.

### Approach used in this notebook
1. Inspect and validate the dataset.
2. Create reproducible train/validation/test splits.
3. Train a **baseline RGB transfer-learning model**.
4. Train a **color-robust model** using strong color augmentation.
5. Evaluate both on normal and deliberately color-shifted test images.
6. Compare accuracy, precision, recall, F1-score and confusion matrices.
7. Save the best model and prediction results.

> **Important:** This notebook is a complete, general solution based on the assessment title visible in the submission form. If the official assessment page specifies a particular dataset, architecture, metric, or experiment, follow that specification and modify the relevant configuration cells accordingly.


## 0. Submission checklist

Before submitting the notebook:

- [ ] All cells run successfully from top to bottom.
- [ ] Dataset path is correct.
- [ ] Final metrics and plots are visible.
- [ ] The conclusion discusses color invariance.
- [ ] The notebook is shared as **Anyone with the link → Viewer**.
- [ ] If a GitHub repository is required by the assessment, add its URL in the final section.
- [ ] Answer the assessment form's AI-use question truthfully.


In [ ]:
# 1. Install / import dependencies
!pip -q install -U scikit-learn seaborn

import os
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 1. Dataset setup

### Expected dataset format

The easiest format is:

```text
dataset/
├── class_1/
│   ├── image001.jpg
│   ├── image002.jpg
│   └── ...
├── class_2/
│   ├── image001.jpg
│   └── ...
└── class_3/
    └── ...
```

If your dataset already contains `train/`, `validation/`, and `test/` folders, the notebook can also be adapted, but the main pipeline below expects a single folder containing one subfolder per class.

### Colab options

**Option A:** Put the extracted dataset at `/content/dataset`.

**Option B:** Upload a ZIP file and let the next cell extract it.


In [ ]:
# 2. Dataset location
DATASET_DIR = Path("/content/dataset")

# If DATASET_DIR does not exist, upload a ZIP file in Colab.
if not DATASET_DIR.exists():
    from google.colab import files
    print("Upload your dataset ZIP file.")
    uploaded = files.upload()
    zip_names = [n for n in uploaded.keys() if n.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("Please upload a .zip dataset.")
    zip_path = Path("/content") / zip_names[0]
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/extracted_dataset")

    # Try to locate the folder containing class directories.
    candidates = []
    for p in Path("/content/extracted_dataset").rglob("*"):
        if p.is_dir():
            subdirs = [x for x in p.iterdir() if x.is_dir()]
            if len(subdirs) >= 2:
                candidates.append(p)

    if candidates:
        DATASET_DIR = candidates[0]

print("Using dataset:", DATASET_DIR)


Upload your dataset ZIP file.


In [ ]:
# 3. Discover classes and image files
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
if len(class_dirs) < 2:
    raise ValueError(
        f"Expected at least 2 class folders inside {DATASET_DIR}. "
        "Please check DATASET_DIR."
    )

class_names = [p.name for p in class_dirs]
class_to_idx = {name: i for i, name in enumerate(class_names)}

records = []
for class_dir in class_dirs:
    label = class_dir.name
    for path in class_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            records.append({"filepath": str(path), "label": label, "label_id": class_to_idx[label]})

df = pd.DataFrame(records)

print("Classes:", class_names)
print("Total images:", len(df))
display(df.head())
print("\nClass distribution:")
display(df["label"].value_counts().sort_index())


In [ ]:
# 4. Visualize class distribution
plt.figure(figsize=(10, 5))
df["label"].value_counts().sort_index().plot(kind="bar")
plt.title("Images per class")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Inspect sample images
n_classes = len(class_names)
fig, axes = plt.subplots(min(n_classes, 4), 3, figsize=(12, 4 * min(n_classes, 4)))
axes = np.array(axes).reshape(-1)

for i, class_name in enumerate(class_names[:4]):
    samples = df[df["label"] == class_name].sample(
        min(3, len(df[df["label"] == class_name])),
        random_state=SEED
    )
    for j, (_, row) in enumerate(samples.iterrows()):
        ax = axes[i * 3 + j]
        img = Image.open(row["filepath"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(class_name)
        ax.axis("off")

for ax in axes[n_classes*3:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 2. Dataset quality checks

The next cells identify corrupted/unreadable images and basic image-size statistics.


In [ ]:
# 6. Check for corrupted images and image dimensions
bad_files = []
sizes = []

for path in df["filepath"]:
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            sizes.append(img.size)
    except Exception as e:
        bad_files.append((path, str(e)))

print("Unreadable/corrupt images:", len(bad_files))
if bad_files:
    display(pd.DataFrame(bad_files, columns=["filepath", "error"]).head(20))

size_df = pd.DataFrame(sizes, columns=["width", "height"])
display(size_df.describe())


## 3. Reproducible train/validation/test split

A stratified split is used so that every class is represented proportionally.


In [ ]:
# 7. Stratified 70/15/15 split
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label_id"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label_id"],
    random_state=SEED
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
display(train_df["label"].value_counts().sort_index())


## 4. TensorFlow input pipeline

Images are resized to 224×224, which is appropriate for EfficientNetB0 transfer learning.


In [ ]:
# 8. Build tf.data datasets from file paths
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(frame, shuffle=False):
    paths = frame["filepath"].values
    labels = frame["label_id"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print("Datasets ready.")


# 5. Baseline model — RGB transfer learning

The baseline uses EfficientNetB0 pretrained on ImageNet.

This establishes a reference point. The important comparison later is whether the color-robust training strategy performs better when test images have their colors changed.


In [ ]:
# 9. Baseline model
NUM_CLASSES = len(class_names)

baseline_model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMG_SIZE, 3)
    ),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.30),
    layers.Dense(NUM_CLASSES, activation="softmax")
], name="baseline_rgb_efficientnet")

# Freeze the pretrained backbone for the initial training stage.
baseline_model.layers[1].trainable = False

baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_model.summary()


In [ ]:
# 10. Train baseline
baseline_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

history_base = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=baseline_callbacks
)


In [ ]:
# 11. Plot baseline learning curves
def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(hist["accuracy"], label="train_accuracy")
    plt.plot(hist["val_accuracy"], label="val_accuracy")
    plt.title(title + " — Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="train_loss")
    plt.plot(hist["val_loss"], label="val_loss")
    plt.title(title + " — Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

plot_history(history_base, "Baseline RGB model")


# 6. Color-robust model

### Why this matters

A classifier can accidentally learn color shortcuts. To make the model more robust, we expose it to many versions of the same design with changed:

- hue
- saturation
- brightness
- contrast
- small geometric transformations

The design structure remains useful while the color information becomes unreliable.

This is a practical way to encourage the model to learn texture, motifs, borders and spatial structure rather than memorizing color.


In [ ]:
# 12. Custom color-jitter augmentation
class ColorJitter(layers.Layer):
    def __init__(self, brightness=0.25, contrast=(0.75, 1.25),
                 saturation=(0.5, 1.5), hue=0.10, **kwargs):
        super().__init__(**kwargs)
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue

    def call(self, images, training=None):
        if not training:
            return images

        # EfficientNet accepts float images in the 0..255 range.
        x = images / 255.0

        x = tf.image.random_brightness(x, max_delta=self.brightness)
        x = tf.image.random_contrast(x, lower=self.contrast[0], upper=self.contrast[1])
        x = tf.image.random_saturation(x, lower=self.saturation[0], upper=self.saturation[1])
        x = tf.image.random_hue(x, max_delta=self.hue)

        x = tf.clip_by_value(x, 0.0, 1.0)
        return x * 255.0


augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    ColorJitter()
], name="color_robust_augmentation")


In [ ]:
# 13. Color-robust model
robust_model = keras.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    augmentation,
    EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMG_SIZE, 3)
    ),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.35),
    layers.Dense(NUM_CLASSES, activation="softmax")
], name="color_robust_efficientnet")

robust_model.layers[2].trainable = False

robust_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

robust_model.summary()


In [ ]:
# 14. Train color-robust model
robust_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

history_robust = robust_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=robust_callbacks
)


In [ ]:
# 15. Plot color-robust learning curves
plot_history(history_robust, "Color-robust model")


# 7. Optional fine-tuning

After the classification head has learned, a small number of final EfficientNet layers can be unfrozen and fine-tuned with a low learning rate.

This often improves transfer-learning performance, but it can also overfit small datasets. The validation set is used to decide whether the fine-tuned version is preferable.


In [ ]:
# 16. Fine-tune the robust model
backbone = robust_model.layers[2]
backbone.trainable = True

# Freeze most layers; fine-tune only the final ~30 layers.
for layer in backbone.layers[:-30]:
    layer.trainable = False

robust_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = robust_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=1,
            min_lr=1e-7
        )
    ]
)


# 8. Standard test-set evaluation

We calculate accuracy, macro precision, macro recall and macro F1-score.


In [ ]:
# 17. Prediction helper
def predict_dataset(model, dataset):
    y_true = []
    y_prob = []

    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_prob.append(probs)
        y_true.extend(labels.numpy().tolist())

    y_true = np.array(y_true)
    y_prob = np.concatenate(y_prob, axis=0)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob

def evaluate_predictions(y_true, y_pred, model_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    }

y_true_base, y_pred_base, y_prob_base = predict_dataset(baseline_model, test_ds)
y_true_robust, y_pred_robust, y_prob_robust = predict_dataset(robust_model, test_ds)

results_normal = pd.DataFrame([
    evaluate_predictions(y_true_base, y_pred_base, "Baseline RGB"),
    evaluate_predictions(y_true_robust, y_pred_robust, "Color-Robust")
])

display(results_normal)


In [ ]:
# 18. Detailed classification report for the robust model
print(classification_report(
    y_true_robust,
    y_pred_robust,
    target_names=class_names,
    digits=4,
    zero_division=0
))


In [ ]:
# 19. Confusion matrix
cm = confusion_matrix(y_true_robust, y_pred_robust)

plt.figure(figsize=(max(8, len(class_names) * 0.7), max(6, len(class_names) * 0.6)))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("Color-Robust Model — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


# 9. Explicit color-shift robustness test

A stronger demonstration of color invariance is to evaluate the models on the **same test images after changing their colors**.

The labels remain unchanged.

If a model is genuinely robust, its performance should degrade less than a color-sensitive baseline.


In [ ]:
# 20. Build a color-shifted version of the test set
def color_shift_image(image):
    # image is float32 0..255
    x = image / 255.0
    x = tf.image.random_brightness(x, max_delta=0.30)
    x = tf.image.random_saturation(x, lower=0.35, upper=1.65)
    x = tf.image.random_hue(x, max_delta=0.15)
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x * 255.0

def make_color_shifted_dataset(frame):
    paths = frame["filepath"].values
    labels = frame["label_id"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_shifted(path, label):
        image, label = load_image(path, label)
        image = color_shift_image(image)
        return image, label

    return ds.map(load_shifted, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

shifted_test_ds = make_color_shifted_dataset(test_df)


In [ ]:
# 21. Evaluate both models under color shift
y_true_base_shift, y_pred_base_shift, _ = predict_dataset(baseline_model, shifted_test_ds)
y_true_robust_shift, y_pred_robust_shift, _ = predict_dataset(robust_model, shifted_test_ds)

results_shifted = pd.DataFrame([
    evaluate_predictions(y_true_base_shift, y_pred_base_shift, "Baseline RGB — Color Shifted"),
    evaluate_predictions(y_true_robust_shift, y_pred_robust_shift, "Color-Robust — Color Shifted")
])

display(results_shifted)


In [ ]:
# 22. Robustness comparison
normal_base_acc = accuracy_score(y_true_base, y_pred_base)
normal_robust_acc = accuracy_score(y_true_robust, y_pred_robust)
shift_base_acc = accuracy_score(y_true_base_shift, y_pred_base_shift)
shift_robust_acc = accuracy_score(y_true_robust_shift, y_pred_robust_shift)

comparison = pd.DataFrame({
    "Model": ["Baseline RGB", "Color-Robust"],
    "Normal Accuracy": [normal_base_acc, normal_robust_acc],
    "Color-Shifted Accuracy": [shift_base_acc, shift_robust_acc],
})

comparison["Accuracy Drop"] = (
    comparison["Normal Accuracy"] - comparison["Color-Shifted Accuracy"]
)

comparison["Robustness Retention"] = (
    comparison["Color-Shifted Accuracy"] / comparison["Normal Accuracy"]
)

display(comparison)


# 10. Visualize color perturbations

This makes the color-invariance idea easy for a reviewer to understand.


In [ ]:
# 23. Show original vs color-shifted images
sample_rows = test_df.sample(min(6, len(test_df)), random_state=SEED)

fig, axes = plt.subplots(len(sample_rows), 2, figsize=(8, 3 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = np.array([axes])

for row_i, (_, row) in enumerate(sample_rows.iterrows()):
    img = tf.io.read_file(row["filepath"])
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)

    shifted = color_shift_image(img)

    axes[row_i, 0].imshow(tf.cast(img, tf.uint8))
    axes[row_i, 0].set_title(f"Original — {row['label']}")
    axes[row_i, 0].axis("off")

    axes[row_i, 1].imshow(tf.cast(shifted, tf.uint8))
    axes[row_i, 1].set_title("Color shifted")
    axes[row_i, 1].axis("off")

plt.tight_layout()
plt.show()


# 11. Error analysis

A good AI-engineering submission should inspect mistakes rather than reporting only accuracy.


In [ ]:
# 24. Show misclassified examples
mis_idx = np.where(y_pred_robust != y_true_robust)[0]

print("Misclassified:", len(mis_idx), "out of", len(y_true_robust))

if len(mis_idx) > 0:
    show_n = min(12, len(mis_idx))
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.ravel()

    # Convert test dataframe to the same order as the dataset.
    test_ordered = test_df.reset_index(drop=True)

    for j, idx in enumerate(mis_idx[:show_n]):
        path = test_ordered.loc[idx, "filepath"]
        img = Image.open(path).convert("RGB")
        axes[j].imshow(img)
        axes[j].set_title(
            f"True: {class_names[y_true_robust[idx]]}\n"
            f"Pred: {class_names[y_pred_robust[idx]]}"
        )
        axes[j].axis("off")

    for ax in axes[show_n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# 12. Save the best model and results

The final model is saved in Keras format. Metrics are saved as CSV files.


In [ ]:
# 25. Save artifacts
OUTPUT_DIR = Path("/content/deeplure_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

robust_model.save(OUTPUT_DIR / "color_robust_saree_model.keras")

results_normal.to_csv(OUTPUT_DIR / "normal_test_metrics.csv", index=False)
results_shifted.to_csv(OUTPUT_DIR / "color_shifted_metrics.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "robustness_comparison.csv", index=False)

with open(OUTPUT_DIR / "class_names.txt", "w") as f:
    for name in class_names:
        f.write(name + "\n")

print("Saved to:", OUTPUT_DIR)


In [ ]:
# 26. Optional: create a ZIP containing final artifacts
zip_output = "/content/deeplure_saree_submission_artifacts.zip"

with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUTPUT_DIR))

print("Created:", zip_output)

# In Google Colab, uncomment to download:
# from google.colab import files
# files.download(zip_output)


# 13. Final results table

Run this section after all training/evaluation cells.


In [ ]:
# 27. Final summary
print("===== FINAL RESULTS =====")
display(results_normal)
display(results_shifted)
display(comparison)

best_row = comparison.loc[comparison["Color-Shifted Accuracy"].idxmax()]

print("\nBest color-shifted model:", best_row["Model"])
print("Color-shifted accuracy:", round(best_row["Color-Shifted Accuracy"], 4))
print("Accuracy retention:", round(best_row["Robustness Retention"], 4))


# 14. Conclusion

Use the actual numbers produced above when writing the final conclusion.

### Suggested conclusion structure

**Objective:**  
The objective was to recognize saree designs while reducing the model's dependence on color.

**Baseline:**  
A pretrained EfficientNetB0 RGB classifier was trained as the baseline.

**Color-robust strategy:**  
The improved model was trained using color perturbations such as brightness, saturation and hue changes, together with standard geometric augmentation. This encourages the model to learn design-related visual information rather than relying on a fixed color appearance.

**Evaluation:**  
Both models were evaluated on the original test set and on a color-shifted test set. The color-shifted evaluation provides a direct test of robustness to color variation.

**Result:**  
Insert the actual accuracy, macro-F1 and robustness-retention values generated by the notebook.

**Limitations:**  
The approach depends on the diversity and quality of the available dataset. Synthetic color changes cannot represent every real-world lighting, fabric, camera or dye variation.

**Future work:**  
Potential improvements include larger and more diverse datasets, domain-specific feature learning, self-supervised pretraining, segmentation of saree regions, and additional real-world color/lighting robustness tests.


# 15. Submission information

Fill these values before submitting the Google Form.

- **Name:** NANNAM SUKUMAR
- **Role:** AI Engineer
- **Assessment:** Color-Invariant Saree Design Recognition
- **Notebook:** This Colab notebook
- **Notebook access:** Anyone with the link → Viewer
- **GitHub repository:** Add only if the official assessment requires it
- **AI tools disclosure:** Answer truthfully according to the tools actually used

### Final check

1. Restart runtime.
2. Run all cells from top to bottom.
3. Confirm there are no errors.
4. Confirm final metrics are visible.
5. Confirm the notebook opens in an incognito window.
6. Submit the notebook link before the stated deadline.
